# EXELU — spike-level analysis, one session

**What this is.** A worked example of the whole analysis on a single recording. It is the
reference workflow: once the choices in here are agreed, the same functions get driven by a
script across all 15 sessions. Nothing in this notebook is specific to one session except the
name in the cell below.

**Where the code lives.** Two plain folders of `.py` files. Nothing to install, nothing to
build — open any of them and read it.

| folder | what is in it |
|---|---|
| `neuroelectrophysiology/scripts/` | reusable on **any** spike-sorted dataset: PSTHs, autocorrelograms, contamination metrics, phase locking, waveform shape, statistics |
| `neuroelectrophysiology/projects/EXELU-spikes/scripts/` | only this experiment: the parameters, the stimulus conditions, the cortex/hippocampus boundary, the figures. All prefixed `EXELU_` |

The split is by *reusability*, not by topic. If a future project needs a PSTH it imports the
same `psth.py`; if it has different stimuli it writes its own `EXELU_triggers.py`.

`autoreload` is on, so editing any of those files takes effect on the next call — no kernel
restart, no re-running the notebook from the top.

**How to read it if you do not write Python.** Every grey block is one step, and is almost
entirely *calls*. To know what a step really does, open the module named in the call — each
one starts with a long plain-English header explaining the idea before any code. Three steps
have a full written derivation:

| step | what it computes | derivation |
|---|---|---|
| the responder metric | which units changed their firing rate | [`methods/responder-metric-calculation.md`](../methods/responder-metric-calculation.md) |
| modulation depth | how strongly a unit follows each stimulus cycle | [`methods/modulation-depth-calculation.md`](../methods/modulation-depth-calculation.md) |
| cell type | principal cell vs interneuron | [`methods/cell-type-classification.md`](../methods/cell-type-classification.md) |

**Vocabulary, once.**

- **unit / cluster** — one putative neuron, as identified by the spike sorter Kilosort.
  Kilosort labels each one `good` (it believes this is a single neuron) or `mua`
  (*multi-unit activity* — several neurons it could not separate).
- **trial / block** — one 4-second presentation of a stimulus. There are 60 of each.
- **PSTH** — *peri-stimulus time histogram*. Line up all 60 trials at the moment the stimulus
  started, count spikes in small time bins, and you can see what the neuron did in response.
- **entrainment / phase locking** — whether spikes arrive at a consistent point within each
  flicker cycle. A different question from whether the firing rate went up, and it needs
  different maths.

## 0 · Setup

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

# Edit any script and the change is live on the next call — no kernel restart.
%load_ext autoreload
%autoreload 2

_project_scripts = next(
    p / "scripts" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "scripts" / "EXELU_paths.py").is_file()
)
sys.path.insert(0, str(_project_scripts))
import EXELU_paths as paths
SHARED_SCRIPTS, PROJECT_SCRIPTS = paths.add_script_paths()

import numpy as np
import pandas as pd

# the session object itself — one per recording, carried through the whole notebook
from ephyslink import Session, load_kilosort

# shared — reusable on any spike-sorted dataset
import entrainment, plotstyle, psth, responders, spiketrains, waveforms
# this project
import EXELU_figures as figures
import EXELU_population as population
import EXELU_records as records
import EXELU_regions as regions
import EXELU_results as results
import EXELU_triggers as triggers
import EXELU_units as units
import EXELU_validation as validation
from EXELU_config import FS, HILL_FP_THRESHOLD_PCT, KS_CONTAM_THRESHOLD_PCT, Params

plotstyle.use_house_style()
%matplotlib inline

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

print(f"shared  scripts : {SHARED_SCRIPTS}")
print(f"project scripts : {PROJECT_SCRIPTS}")

### The one thing to change

`SESSION_NAME` picks the recording. `params` holds every analysis choice; the defaults and the
reason for each are in `scripts/EXELU_config.py` alongside this project. To try something different,
do not edit the file — override it here, so the notebook stays a record of what was actually
run:

```python
params = replace(params, psth_bin_s=0.010)   # everything else unchanged and still stated
```

In [ ]:
SESSION_NAME = "2026-04-14_11-46-55"

params = Params()

print(f"data root  : {paths.data_root()}")
print(f"sessions   : {len(paths.session_names())} available")
print(f"output to  : {paths.output_dir(SESSION_NAME)}")

## 1 · Load the session

### Loading

`ephyslink.load_kilosort` reads the sorter's output into a **`SessionKS`** — the one object
that carries through this entire notebook. Three EXELU-specific things are attached to it
here rather than inside a loader, so the assembly is visible in the worked example:

1. `session_particulars.txt` — the anatomical landmark channels, plus the animal identity
   written in by `EXELU_update_particulars.py`;
2. the matching row of `mice-records`, as a fallback for any field the particulars lack;
3. `analog_events.mat` — the stimulus TTLs.

Everything added **after** `load_kilosort` returns is analysis output by definition: the
loader records what it produced, which is what lets the results export at the end strip the
raw data without being told what the raw data was.

*(The future batch script repeats these few lines. That is the cost of keeping the assembly
in the notebook, and it is small because each line is a call to a named function.)*

In [ ]:
session_path = paths.session_dir(SESSION_NAME)

session = load_kilosort(session_path, session_id=SESSION_NAME, experiment="EXELU", fs_hz=FS)

session.meta["particulars"], _ = records.parse_particulars(
    session_path / records.PARTICULARS_FILENAME
)
session.meta["mice_record"] = records.record_for_session(SESSION_NAME)
for channel, samples in triggers.read_analog_events(session_path).items():
    session.add_events(channel, samples)

regions.check_pitch(session)     # the CTX/HPC boundary depends on the contact spacing
session.mark_source()            # everything from here on is analysis output

store = spiketrains.SpikeStore(session.array("spike_times"),
                               session.array("spike_clusters"), FS)

print(records.describe(session))

### Depth to region

The probe is one shank going down through visual cortex into hippocampus. The only anatomy
available is the set of landmark channels the experimenter marked from the LFP. The
cortex/hippocampus boundary is taken as `thetaCh` — the depth at which the theta rhythm
appears.

Channel numbering is **1-based**, confirmed by the professor, so channel *n* sits at
`(n − 1) × 20 µm`. Getting this wrong shifts the boundary by one contact and has previously
moved units from one region to the other.

In [ ]:
print(f"landmarks (µm)          : { {k: round(v) for k, v in regions.landmarks(session).items()} }")
print(f"CTX/HPC boundary        : thetaCh {regions.particulars(session)['thetaCh']} "
      f"→ {regions.boundary(session):.0f} µm")

## 2 · Stimulus triggers

Everything below is anchored to a moment when the stimulus changed. Three traps are handled
explicitly in `exelu/triggers.py`, and a fourth thing is checked rather than assumed:

1. **Calibration events** at 283–348 s, before the protocol starts at ~370 s, are dropped by
   requiring a block to contain exactly the expected number of transitions.
2. **Static events are ON/OFF pairs.** An ON is one whose partner follows 4.000–4.002 s
   later; a loose tolerance would fold stimulus-*offset* responses into the onset histogram.
3. **Contact bounce** on the static line is removed first.
4. **Conditions interleave**, so one condition's baseline could land inside another's block.
   Asserted, not assumed.

**Review point 5** is applied here: `phase_reversing_2` is excluded from everything. It is
the opposite edge polarity of the same physical grating reversal as `phase_reversing_1` —
identical 160 transitions per block, offset by 12.5 ms — so keeping both counts one stimulus
twice.

In [ ]:
trigger_set = triggers.build_triggers(session.events, params)
CONDITIONS = list(trigger_set)
FLICKER_CONDITIONS = triggers.flicker_conditions(trigger_set)

print(f"excluded: {params.excluded_conditions}\n")
print(triggers.summarise(trigger_set).to_string(index=False))

gap = triggers.check_baseline_clear(trigger_set, params.psth_pre_s)
print(f"\ntightest stimulus-free gap before any block onset: {gap:.3f} s "
      f"— the {params.psth_pre_s:.0f} s baseline is clean")

## 3 · Units, and how contaminated they are

### What the two contamination numbers mean — review point 4

A real neuron cannot fire twice within its refractory period (~2.5 ms). Spikes at short
lags therefore came from *somewhere else*, and counting them is how cluster quality is
measured. Two established ways to count them, and they are **not the same quantity**:

| | Kilosort `ContamPct` | Hill et al. (2011) $f_p$ |
|---|---|---|
| what it is | refractory-window spike **density** ÷ the unit's own baseline density | estimated **fraction of spikes** in the cluster that do not belong |
| range | 0 % to unbounded | 0 % to 50 %, then undefined |
| 100 % means | refractory window as full as baseline | — |
| \>100 % | common; it means the unit **bursts** | impossible |
| threshold | **20 %** — Kilosort's own | **10 %** — the usual convention |

The 20 % figure is not taken on faith. Across all 15 sessions the largest `ContamPct` on
any `good` cluster is 17.7–20.0 % and the smallest on any `mua` cluster is 20.0 %, so `good`
implies `ContamPct < 20 %`. (The converse does not hold — Kilosort also rejects on spike
count and amplitude, so `mua` clusters with 0 % contamination exist.)

Because the two metrics are different quantities, each histogram below is coloured against
**its own** threshold. Green bars are the clusters that metric accepts. Bin edges are forced
to land on the threshold so that no bar is part-accepted and part-rejected.

In [ ]:
unit_table = units.build_unit_table(session, store, params, FS)
session.add_table("units_qc", unit_table)     # every cluster, not just the clean ones

print(unit_table.groupby("label").agg(
    clusters=("unit", "size"),
    median_contam_pct=("contam_pct", "median"),
    median_fr_hz=("fr_hz", "median"),
).round(2).to_string())

**Guard.** Kilosort's `ContamPct` is re-implemented from `kilosort/CCG.py` because no library
computes it. The only way to know the re-implementation is the same number is to check it
against the sorter's own output file, to within the one decimal place that file is written
to.

In [ ]:
guard = validation.contamination_guard(session, store)
print(f"ContamPct re-implementation vs cluster_ContamPct.tsv: "
      f"max absolute error {guard.abs_error.max():.3f} percentage points "
      f"over {len(guard)} clusters")

In [ ]:
saver = plotstyle.FigureSaver(paths.figure_dir(session.id))

fig = figures.plot_contamination_comparison(
    session, unit_table, params, KS_CONTAM_THRESHOLD_PCT, HILL_FP_THRESHOLD_PCT
)
saver(fig, "contamination_ks_vs_hill")

### The analysis set

`clean` is what every later step runs on: Kilosort `good`, contamination under the ceiling,
and enough spikes to say anything. The ceiling defaults to Kilosort's own 20 %, which makes
the filter redundant with the `good` label by design — the set is then exactly "what
Kilosort called a single unit". The stricter 10 % count is printed alongside so the cost of
that choice is visible rather than buried.

In [ ]:
clean = units.select_clean(unit_table, params)
strict = units.select_clean(unit_table, replace(params, clean_contam_pct=10.0))

print(f"at {params.clean_contam_pct:.0f}% (Kilosort's own threshold) : {len(clean):>3} units  "
      f"{dict(clean.region.value_counts())}")
print(f"at 10% (stricter convention)                  : {len(strict):>3} units  "
      f"{dict(strict.region.value_counts())}")

dropped = clean[~clean.unit.isin(strict.unit)]
print(f"\nthe {len(dropped)} units the stricter rule would remove:")
print(dropped[["unit", "region", "depth_um", "fr_hz", "contam_pct", "n_spikes"]]
      .round(2).to_string(index=False))

## 4 · Cell type — review point 2

Two populations are expected, with three signatures each:

|  | waveform | firing rate | autocorrelogram |
|---|---|---|---|
| **principal cells** | wider | lower | bursty, theta-modulated |
| **interneurons** | narrow | high | not bursty, gamma-range |

The label is assigned on **waveform width alone** — trough-to-peak duration against a fixed
0.45 ms threshold. Firing rate, burst index and theta index are measured independently and
used only to *flag* units whose physiology contradicts their waveform. A multi-feature
classifier would give a tidier figure and a label nobody could interrogate.

Three honest limitations, all structural:

- The raw voltage traces are not available, so the waveform comes from Kilosort's
  **templates**, not from an average of raw snippets. Templates are smoother than true mean
  waveforms.
- Templates are un-whitened with `whitening_mat_inv` first, as Phy does. The whitening
  matrix here is 99.4 % diagonal so the effect on shape is small, but it can change which
  channel is the peak channel.
- "Gamma-modulated" cannot be measured: there is no LFP in this dataset, so the theta index
  is *rhythmicity of the unit's own spike train*, not phase-locking to a measured
  oscillation.

In [ ]:
clean = units.add_cell_types(session, store, clean, params, FS)
session.add_table("units", clean)          # ← attached; it travels with the session now

dips = waveforms.kde_antimodes(clean.trough_to_peak_ms.values)
print(f"split used                     : {params.t2p_split_ms:.2f} ms (literature default)")
print(f"prominent dips in this session : {np.round(dips, 3)} ms")
print()
print(clean.groupby(["cell_type", "region"]).size().unstack(fill_value=0).to_string())
print()
print("median metric per class:")
print(clean.groupby("cell_type")[
    ["trough_to_peak_ms", "half_width_ms", "fr_hz", "burst_index", "theta_index"]
].median().round(3).to_string())

print(f"\n{int((~clean.cell_type_consistent).sum())} unit(s) flagged: physiology disagrees with "
      f"the waveform, or the waveform is positive-dominant")
print(f"{int(clean.positive_dominant.fillna(False).sum())} positive-dominant · "
      f"{int((~clean.repolarises.fillna(False).astype(bool)).sum())} never repolarise, so "
      f"trough-to-peak is undefined for them and they are `unclassified`")
print(f"{int((~clean.rhythmicity_reliable).sum())} unit(s) fire too rarely for the rhythmicity "
      f"indices to mean anything (reported as NaN, not as a large number)")

In [ ]:
fig = figures.plot_celltype_summary(session, clean, params, dips)
saver(fig, "cell_type_summary")

## 5 · Autocorrelograms — review points 2 and 3

An autocorrelogram histograms the time differences between every pair of spikes from one
unit. Read at two window widths it answers two unrelated questions.

**±50 ms — is this one neuron?** The count must fall to near zero inside the ±2.5 ms
refractory window (marked in soft red). A filled-in centre means the cluster is a merge.
The 25 ms marks are one 40 Hz cycle.

Each panel now carries the session, animal and genotype in the title block, the cell-type
call, and an inset of the unit's waveform so the classification can be checked by eye.

In [ ]:
fig = figures.plot_acg_grid(session, store, clean, params)
saver(fig, "acg_refractory")

**±500 ms — is this unit theta-modulated?** Theta is 4–12 Hz, a period of 83–250 ms. The
±50 ms window above cannot show even one theta cycle, which is exactly why review point 3
asks for a second, wider one. A theta-modulated unit shows a side-peak in the green band
(100–140 ms, one theta period) and a dip in the red band (50–70 ms, half a period); the
theta index is built from precisely those two windows.

In [ ]:
fig = figures.plot_acg_theta_grid(session, store, clean, params)
saver(fig, "acg_theta")

## 6 · Block PSTH, standardised per trial — review point 6

The mouse is not in the same state on every trial. When it is moving, cortical firing is
already elevated *before* the stimulus arrives. Averaging the raw trials and subtracting one
grand baseline leaves that offset inside the average, so a session with more movement looks
like a session with more stimulus drive.

The fix is to remove **each trial's own** pre-stimulus mean before averaging:

$$
d_{t,b} = r_{t,b} - \frac{1}{|B|}\sum_{b' \in B} r_{t,b'}
\qquad
a_b = \frac{1}{T}\sum_t d_{t,b}
\qquad
z_b = \frac{a_b}{\sigma}
$$

with $r_{t,b}$ the rate of the unit in trial $t$, bin $b$; $B$ the baseline bins; and
$\sigma$ the standard deviation of $a$ over the baseline bins, floored at the Poisson
expectation.

**Why not the literal per-trial z-score.** Dividing each trial by its own baseline SD is
undefined for most trials of most units here — a 0.5 Hz unit fires 0 or 1 spikes in its 2 s
baseline, so that SD is 0. The scale therefore comes from one pooled estimate; only the
*offset* is removed per trial, which is the part that movement actually changes.

First, the evidence that this matters at all.

In [ ]:
clean_spikes = store.as_dict(clean.unit)
onsets = triggers.onsets_by_condition(trigger_set)

spread = units.attach_metadata(
    psth.trial_baseline_spread(
        clean_spikes, onsets, params.psth_pre_s, params.psth_post_s, params.psth_bin_s, FS
    ),
    clean,
)
session.add_table("baseline_spread", spread)

print(f"per-trial baseline coefficient of variation — median {spread.baseline_cv.median():.2f}, "
      f"upper quartile {spread.baseline_cv.quantile(0.75):.2f}")
print("a CV near 1 means a unit's baseline rate routinely varies by ~100% from trial to trial,")
print("so the per-trial correction is doing real work rather than tidying a rounding error.")

fig = figures.plot_baseline_variability(session, spread, params.psth_pre_s)
saver(fig, "baseline_variability")

In [ ]:
# cortex block above, hippocampus below; descending firing rate within each
ordered, n_ctx = units.order_by_region_then_rate(clean)
ordered_spikes = store.as_dict(ordered.unit)

centres_by_condition, z_by_condition = {}, {}
for condition in CONDITIONS:
    centres, z, _, _ = psth.block_psth_matrix(
        ordered_spikes, ordered.unit, onsets[condition],
        params.psth_pre_s, params.psth_post_s, params.psth_bin_s, FS,
    )
    centres_by_condition[condition], z_by_condition[condition] = centres, z

fig = figures.plot_block_psth_heatmap(
    session, centres_by_condition, z_by_condition, ordered, n_ctx, params
)
saver(fig, "block_psth_clean")

## 7 · The responder metric — review point 7  ·  DEEP DIVE

> Full derivation, the alternatives, and the failure modes:
> [`methods/responder-metric-calculation.md`](../methods/responder-metric-calculation.md)

Per unit and per condition:

1. **Trial-wise rates** — spikes in the 2 s before onset and the 4 s of stimulus, for each
   of the ~60 trials. Rates rather than counts, because the two windows differ in length.
2. **z-scored against the unit's own baseline trials** — so a 30 Hz interneuron and a 0.3 Hz
   granule cell land on the same scale, and "z = 2" means *two of this unit's own baseline
   standard deviations*, not two of the population's.
3. **Paired Wilcoxon signed-rank, pre versus post, within condition** — paired because the
   same trial supplies both numbers, non-parametric because trial-wise counts are neither
   normal nor equal-variance.
4. **Benjamini–Hochberg across units** — 45 units tested at α = 0.05 would give two false
   positives from noise alone.
5. A **sign-flip permutation test** confirms the Wilcoxon, since it assumes strictly less.

The professor's earlier rule — z of the population-relative % change above 2 SD — is
reported as `responder_legacy`, not as the headline. Its reference distribution is *the
other units*, which silently assumes most units do not respond and inflates the SD by the
very effect being looked for.

In [ ]:
resp = units.attach_metadata(
    responders.responder_table(
        clean_spikes, onsets, params.psth_pre_s, params.psth_post_s, FS,
        alpha=params.responder_alpha, n_permutations=params.responder_n_permutations,
    ),
    clean,
)
# the professor's original population-relative rule, kept for comparison only
resp["responder_legacy"] = resp.z_pct_population.abs() > params.responder_z_legacy
session.add_table("responders", resp)
session.log("responder test", alpha=params.responder_alpha,
            n_permutations=params.responder_n_permutations)

print(resp.groupby(["condition", "region"]).agg(
    units=("unit", "size"),
    responders=("responder", "sum"),
    up=("direction", lambda s: int((s == "up").sum())),
    down=("direction", lambda s: int((s == "down").sum())),
    legacy_z2sd=("responder_legacy", "sum"),
    median_z_stim=("z_stim", "median"),
    median_pct_change=("pct_change", "median"),
).round(3).to_string())
print()
print("do the Wilcoxon and the permutation test agree?")
print(responders.test_agreement(resp).to_string(index=False))

In [ ]:
fig = figures.plot_responders(session, resp, params)
saver(fig, "responders_by_depth")

## 8 · Transition-triggered PSTH and modulation depth — review point 9  ·  DEEP DIVE

> Full derivation, exactly how peak and trough are detected, and the folding artifact that
> used to inflate every number:
> [`methods/modulation-depth-calculation.md`](../methods/modulation-depth-calculation.md)

This asks a different question from section 6. Not *did the block drive the unit?* but
*does the unit follow each individual screen transition?* Every transition is a trigger —
9 600 of them at 40 Hz — and the histogram is binned at 1 ms, twenty-five bins per 40 Hz
cycle.

Two things to hold in mind when reading the panels:

- **The displayed 3.5 cycles overlap.** Triggers are one cycle apart and the window is 3.5
  cycles wide, so each spike appears in about three and a half windows. The rate axis is
  correct, but the histogram is periodic *by construction* — the informative object is a
  single cycle. **No number is measured from the display array**; the metrics come from a
  one-cycle histogram in which each spike is counted exactly once.
- **A flat histogram is the null.** It does not mean the unit is silent; it means unmodulated.

**Guard first.** Three synthetic spike trains with known answers go through the identical
estimator. This exists because a previous version folded a 3.5-cycle window modulo the
period — and 3.5 is not a whole number, so a perfectly flat train read a modulation depth of
1.33 from geometry alone.

In [ ]:
print(validation.modulation_depth_guard(params).to_string(index=False))
print()
print("The flat train reads slightly above 1 rather than exactly 1. That residual is the")
print("order-statistic bias: with finite counts the maximum of 25 bins sits above the mean and")
print("the minimum below it, so peak/trough exceeds 1 even with no modulation. It is the reason")
print("`mod_depth` is not comparable between units of different firing rate, and the reason")
print("`ppc` exists.")

In [ ]:
transitions = triggers.transitions_by_condition(trigger_set)
periods = triggers.periods_by_condition(trigger_set)

tt = units.attach_metadata(
    entrainment.transition_table(
        clean_spikes, transitions, periods, FS,
        cycles=params.tt_cycles, bin_ms=params.tt_bin_ms,
        smooth_bins=params.tt_smooth_bins, min_spikes=params.tt_min_spikes,
    ),
    clean,
)
session.add_table("entrainment", tt)
session.log("entrainment", bin_ms=params.tt_bin_ms, min_spikes=params.tt_min_spikes)

print(tt[tt.reliable].groupby(["condition", "region"]).agg(
    units=("unit", "size"),
    locked=("locked", "sum"),
    median_ppc=("ppc", "median"),
    median_mod_depth=("mod_depth", "median"),
    median_mod_depth_norm=("mod_depth_norm", "median"),
).round(4).to_string())

In [ ]:
for condition in FLICKER_CONDITIONS:
    fig = figures.plot_transition_grid(session, store, ordered, trigger_set[condition], params)
    saver(fig, f"transition_grid_{condition}")

### Phase locking against depth

The claim the whole probe design exists to test: does 40 Hz flicker entrain spiking in
cortex and fail to reach hippocampus? **PPC** rather than vector strength, because the two
regions differ systematically in firing rate and vector strength is biased upward at low
spike count — a raw vector-strength comparison would partly be a firing-rate comparison.

In [ ]:
fig = figures.plot_locking_vs_depth(session, tt, params)
saver(fig, "locking_vs_depth")

### Harmonic control — is weak 40 Hz locking real, or cancelled?

Vector strength has a known blind spot: a unit firing **twice per cycle at opposite phases**
has its two phase vectors cancel, and reads ≈ 0 despite perfect entrainment. The analog line
logs one edge per cycle while the display physically changes twice, so this is a live
possibility rather than a hypothetical.

The diagnostic is **not** "which harmonic is larger on average" — for an unmodulated unit
both are noise and that comparison is a coin flip carrying no information. It is whether any
unit is significantly locked at $2f_0$ while *not* locked at $f_0$, which is the only
signature of the cancellation.

In [ ]:
harmonics = units.attach_metadata(
    entrainment.harmonic_table(clean_spikes, transitions, periods, FS,
                               min_spikes=params.tt_min_spikes),
    clean,
)
session.add_table("harmonic_control", harmonics)

for condition in harmonics.condition.unique():
    subset = harmonics[(harmonics.condition == condition) & harmonics.reliable]
    print(f"{condition:<18} locked at f0: {int((subset.q_f0 < 0.05).sum()):>2}/{len(subset)}   "
          f"locked at 2f0 ONLY: {int(subset.locked_at_2f0_only.sum())}")

fig = figures.plot_harmonic_control(session, harmonics)
saver(fig, "harmonic_control")

## 9 · Multi-unit activity — review point 8

Two stages, in this order.

**9a — every `mua` cluster separately**, ordered shallow to deep, with the anatomical
landmarks drawn on. No contamination filter is applied: being contaminated is what the `mua`
label *means*.

In [ ]:
mua = (unit_table[(unit_table.label == "mua") & (unit_table.n_spikes >= params.min_spikes_unit)]
       .sort_values("depth_um").reset_index(drop=True))
mua_n_ctx = int((mua.region == "CTX").sum())
mua_spikes = store.as_dict(mua.unit)

mua_centres, mua_z = {}, {}
for condition in CONDITIONS:
    centres, z, _, _ = psth.block_psth_matrix(
        mua_spikes, mua.unit, onsets[condition],
        params.psth_pre_s, params.psth_post_s, params.psth_bin_s, FS,
    )
    mua_centres[condition], mua_z[condition] = centres, z

fig = figures.plot_mua_depth_heatmap(session, mua_centres, mua_z, mua, mua_n_ctx, params)
saver(fig, "mua_block_psth_individual")

**9b — pooled per region.** Every cortical `mua` spike merged into one train, every
hippocampal one into another, and the block PSTH taken over exactly $[-2\,\mathrm{s},
+4\,\mathrm{s}]$.

What pooling costs, and it belongs in the report:

- a pooled train has **no refractory period**, so every contamination diagnostic is
  meaningless for it and none is computed;
- the pool is **dominated by its highest-rate members**, so it is closer to "population rate
  near these contacts" than to "the average neuron". A large pooled modulation is consistent
  with a few strongly driven units rather than with broad engagement, and the two cannot be
  told apart from the pooled trace alone — which is why 9a is plotted as well;
- whether `good` clusters join the pool is a real choice, exposed as
  `params.pool_include_good` and defaulting to *no*.

In [ ]:
pooled = population.register_pools(store, unit_table, params)
print(pooled.to_string(index=False))

regulation = population.regulation_table(session, store, pooled, trigger_set, params, FS)
session.add_table("pooled_mua_regulation", regulation)

print()
print(regulation[["region", "condition", "n_trials", "fr_pre_hz", "fr_stim_hz",
                  "delta_hz", "pct_change", "mod_index", "peak_z",
                  "p_wilcoxon", "p_permutation"]].round(4).to_string(index=False))

In [ ]:
fig = figures.plot_pooled_block_psth(session, store, pooled, trigger_set, regulation, params)
saver(fig, "mua_pooled_block_psth")

fig = figures.plot_pooled_transition(session, store, pooled, trigger_set, params)
saver(fig, "mua_pooled_transition")

**The same two analyses on the multi-unit data.** Individual `mua` clusters and the two
pooled trains go through the identical responder and entrainment functions, and are attached
to the session alongside the single-unit tables. They travel into the results file, so a
cross-session comparison can ask the multi-unit question without the raw data.

In [ ]:
pooled_spikes = store.as_dict(pooled.unit)

session.add_table("responders_mua", units.attach_metadata(
    responders.responder_table(mua_spikes, onsets, params.psth_pre_s, params.psth_post_s, FS,
                               n_permutations=0), mua))
session.add_table("entrainment_mua", units.attach_metadata(
    entrainment.transition_table(mua_spikes, transitions, periods, FS,
                                 min_spikes=params.tt_min_spikes), mua))
session.add_table("responders_pooled", responders.responder_table(
    pooled_spikes, onsets, params.psth_pre_s, params.psth_post_s, FS))
session.add_table("entrainment_pooled", entrainment.transition_table(
    pooled_spikes, transitions, periods, FS, min_spikes=params.tt_min_spikes))

for name in ["responders_mua", "entrainment_mua", "responders_pooled", "entrainment_pooled"]:
    print(f"    {name:<22} {session.tables[name].shape}")

**The pooled magnitudes also go to a flat cross-session CSV.** That predates the results
export below and is kept because it is the one thing a spreadsheet can open directly. It is
the same numbers as `pooled_mua_regulation` inside the results file.

In [ ]:
cross_session = population.append_across_sessions(regulation)
print(f"cross-session table: {cross_session}")
print(pd.read_csv(cross_session)[["session", "genotype", "region", "condition",
                                  "pct_change", "mod_index", "peak_z"]]
      .round(3).to_string(index=False))

## 10 · Export — the session, and the results

Two files come out of this notebook, and they are for different readers.

**The full session** (`output/<session>/<session>.h5`, ~44 MB) is the working object: raw
spikes, templates and contacts *plus* every table this analysis attached. Reload it to pick
the analysis up where it left off, or open it in Julia with the same axis order.

**The results file** (`output/_results/<session>.h5`, ~200 KB) is what a comparison across
sessions reads. It keeps every derived table at full per-row resolution and a flat dict of
session-level numbers, and drops every spike, template and waveform. Fifteen of these fit in
memory; fifteen full sessions do not.

The split needs no list of what to keep. `load_kilosort` recorded what it produced, so
"results" is simply "everything added since" — which is why `mark_source()` was called right
after loading.

In [ ]:
# ---- the session-level numbers, flattened from the tables above ----------------
session.results.update(results.summarise(session))
print(f"{len(session.results)} summary numbers, e.g.")
for key in ["genotype", "n_clean", "n_locked_flicker_20hz_ctx", "median_ppc_flicker_20hz_ctx",
            "n_responders_flicker_40hz_ctx", "pooled_pct_change_flicker_20hz_hpc"]:
    print(f"    {key:<34} {session.results.get(key)}")

# ---- CSVs, for anything that wants a spreadsheet --------------------------------
# Written from session.tables rather than from the local variables, so there is one source
# of truth: what the CSV says and what the results file says cannot drift apart.
print("\nCSV:")
for name in session.derived_tables():
    path = paths.output_dir(session.id) / f"{name}.csv"
    session.tables[name].to_csv(path, index=False)
    print(f"    {name:<24} {str(session.tables[name].shape):<12} -> {path.name}")

# ---- the two session files ------------------------------------------------------
full_path = session.save(paths.session_file(session.id))
results_path = results.export(session)

print(f"\nfull session   {full_path.stat().st_size / 1e6:6.1f} MB   {full_path.name}")
print(f"results only   {results_path.stat().st_size / 1e3:6.0f} KB   {results_path.name}")
print(f"\n{results.check_exported(session, results_path)}")
print(f"\n{saver.count} figures -> {paths.figure_dir(session.id)}")

### Reading the results back

This is all a comparison notebook needs — no raw data, no Kilosort directory.

`read_results` stacks each named table across every session it finds, adding a `session`
column, and builds one summary row per session from the `results` dict.

In [ ]:
from ephyslink import read_results

comparison = read_results(paths.results_dir())
print(comparison)
print()
print("summary, one row per session (a few of the columns):")
print(comparison.summary[["genotype", "n_clean", "n_locked_flicker_20hz_ctx",
                          "median_ppc_flicker_20hz_ctx"]].to_string())
print()
print("stacked tables, ready to pool across sessions once the batch has run:")
for name, frame in comparison.tables.items():
    print(f"    {name:<24} {len(frame):>4} rows × {len(frame.columns)} columns")

## 11 · Open questions for the professor

Ranked by how much each could change a conclusion.

1. **`photodiode` is bit-identical to `flicker_20Hz`** — the same 4 805 timestamps, verified
   by array equality. Either the channel map is wrong or the photodiode covered only one
   condition. Without an independent optical trace there is no check that the analog TTLs
   coincide with actual screen changes and no measure of display latency, so **no
   absolute-latency or preferred-phase claim is defensible.** Blocking for latency only.
2. **Nothing follows the phase-reversing grating** although the same units follow flicker.
   Is that TTL marking what we think it is?
3. **20 Hz drives cortical spiking harder than 40 Hz**, on both the responder count and PPC.
   20 Hz is the nominal *control* frequency. How should this be framed, given the published
   40 Hz result is an LFP-power result rather than a spike-locking one?
4. **`mice-records` has a `settle_time` column** (15 for these sessions) that is not used
   here, because its unit is ambiguous — 15 seconds is implausibly short and 15 minutes would
   discard the first stimulus blocks. `params.settle_s` is still the assumed 300 s. Confirm
   the unit and the intended cutoff.
5. **The isolated events at 283–348 s** in `static`, `flicker_40Hz` and `flicker_20Hz` —
   single transitions, seconds apart, before the protocol proper. Assumed calibration and
   dropped. Confirm.
6. **The analysis set now uses Kilosort's own 20 % contamination threshold** rather than the
   stricter 10 % used in the previous report, which changes the clean set from 35 units to
   45 and therefore moves every headline count. Confirm which is wanted.
7. **A shuffle control is still not implemented**, as instructed. It remains the correct null
   for the responder question — random triggers drawn from the non-exposure periods would give
   a per-unit null with the right false-positive rate by construction, replacing both the
   population z-score and the parametric assumptions of the Wilcoxon.